# Module 3 · Reading what my detector produces
This notebook uses **your** `src/evaluate.py` with the model you trained on the page *2 · Training*. Change `RUN` to the name of your run. It writes the figures your Assignment 3.2 needs.

In [ ]:
import json, sys
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
ROOT = Path("..") if Path("../src").exists() else Path(".")
sys.path.insert(0, str(ROOT))
from src import data as dt, evaluate as ev

RUN = "my_run"                       # the name of your training run
project = dt.load_project(ROOT / "project.yaml")
classes = project["classes"]
data_root = ROOT / "data" / f"{project['name']}_yolo"
weights = ROOT / "runs" / RUN / "weights" / "best.pt"
yaml_path = data_root / "data.yaml"
(ROOT / "results").mkdir(exist_ok=True)
print(weights, weights.exists())

## 1. The score on photos the model never saw
mAP goes through all the confidence values, so it has no threshold of its own.

In [ ]:
metrics = ev.yolo_metrics(weights, yaml_path)
print(json.dumps(metrics, indent=1))

## 2. The same photos at four thresholds

In [ ]:
images = ev.predict_images(weights, data_root / "images" / "test", data_root / "labels" / "test", classes)
table = ev.threshold_table(images)
for r in table:
    print(f"conf {r['conf']:.2f} · TP {r['tp']:4d} · FP {r['fp']:4d} · FN {r['fn']:4d} · precision {r['precision']:.2f} · recall {r['recall']:.2f}")
print("from 0.25 to 0.75:", ev.what_changed(images, 0.25, 0.75))
json.dump(table, open(ROOT / "results" / "thresholds.json", "w"), indent=1)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([r["conf"] for r in table], [r["precision"] for r in table], "-o", label="precision")
ax.plot([r["conf"] for r in table], [r["recall"] for r in table], "-o", label="recall")
ax.set_xlabel("confidence threshold"); ax.set_ylim(0, 1); ax.legend(); fig.tight_layout()
fig.savefig(ROOT / "results" / "thresholds.png", dpi=150); plt.show()

## 3. Look at the images
Green: correct. Red: false alarm. Yellow: an object the detector missed.

In [ ]:
def panel(name):
    rgb = cv2.cvtColor(cv2.imread(str(data_root / "images" / "test" / name)), cv2.COLOR_BGR2RGB)
    d = images[name]
    return ev.draw_result(rgb, ev.match([p for p in d["preds"] if p["score"] >= 0.25], d["truths"]))
picked = ev.pick_examples(images, conf=0.25, n=3)
names = picked["correct"] + picked["wrong"]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, n in zip(axes.flat, names):
    ax.imshow(panel(n)); ax.set_title(n, fontsize=9)
for ax in axes.flat: ax.axis("off")
fig.tight_layout(); fig.savefig(ROOT / "results" / "examples.png", dpi=150); plt.show()
print(picked)

## 4. Now it is your turn
Ask the subagent `image-auditor` to look at `results/examples.png`, and `metrics-reviewer` to read your draft. They point things out. **The interpretation is yours.**